## Progettazione di un Data Warehouse
Progettare un data warehouse per la gestione delle problematiche illustrate nei punti seguenti relative all'esportazione dei vini italiani per il progetto Made In Italy.

### Descrizione del problema
Nell’ambito della valorizzazione delle eccellenze del Made in Italy, il Governo italiano vuole analizzare le **esportazioni** di vini italiani all’estero, per capire, ad esempio, quali sono i vini più richiesti, da quali zone d’Italia provengono e quali sono i principali Stati esteri importatori.
Il Governo ha raccolto dalle aziende vinicole italiane l’elenco degli ordini di acquisto. Ogni ordine di acquisto è caratterizzato da data dell’ordine, importo in euro, quantità in litri, tipo di vino, tipo di confezionamento (es. bottiglia, cartone, damigiana, ecc.), numero di partita IVA dell’azienda vinicola di provenienza, stato estero di destinazione. 
Gli analisti, a partire dagli ordini di acquisto, vogliono creare un data warehouse che sintetizzi i dati raccolti per rispondere ad un serie di interrogazioni.
In fase di realizzazione, gli analisti integrano i dati degli ordini con dei dati di contesto in loro possesso, necessari per svolgere le analisi di interesse. In particolare, ad ogni tipo di vino viene associato l’elenco delle certificazioni di qualità (DOC, DOP, o DOCG) che possiede. Per esempio: il tipo di vino “Barolo” è sempre associato alla certificazione DOCG, il tipo di vino “Nebbiolo” è sempre associato alle certificazioni DOP e DOCG, ecc. Ogni certificazione può essere associata a più tipi di vino. Alcuni tipi di vino possono essere senza certificazioni. Ad ogni ordine è, inoltre, associata la sua dimensione: sono considerati piccoli gli ordini fino a 100 litri, medi quelli fino a 1000 litri, e grandi quelli di quantità superiore. Infine, per ogni azienda vinicola, è possibile risalire dal numero di partita IVA ai
dati anagrafici completi (denominazione, indirizzo, provincia, regione, area geografica della regione). Le interrogazioni cui sono interessati gli analisti riguardano la ***quantità totale*** di vino esportata (in litri), e il ***prezzo*** medio per litro, al variare di:
• *mese, bimestre, trimestre, quadrimestre, semestre e anno*;
• *provincia, regione e area geografica (Nord, Centro, Sud)* dell’azienda vinicola;
• *tipo di vino*;
• *certificazioni di qualità*;
• *tipo di confezionamento*;
• *stato estero di destinazione, e relativo continente*;
• *dimensione dell’ordine (piccolo, medio, grande)*;
Il data warehouse traccerà le informazioni relative agli anni dal 2010 al 2013. 

#### Di seguito sono riportate alcune interrogazioni specifiche:
1. Considerando solo i vini che hanno la certificazione “DOC” esportati in Asia, selezionare il prezzo medio al litro per ogni anno, la percentuale di litri esportati nell’anno rispetto al totale di tutti gli anni, e il totale cumulativo annuale di litri esportati. Si efettui l’analisi separatamente per ogni tipo di confezionamento.
2. Per ogni regione, selezionare il prezzo medio al litro, il numero medio di litri di vino esportati per provincia, e la percentuale di litri di vino esportati da ogni regione rispetto al totale dell’area geografica di appartenenza (Nord, Centro, Sud). Si efettui l’analisi separatamente per ogni anno.
3. Assegnare un rank ad ogni stato estero per quantità totale di vino importata. Efettuare l’analisi separatamente per ogni tipo di vino. Si considerino solo gli ordini di grandi dimensioni.


## Progettazione Concettuale
### Facts:
- #### Esportazioni

![SchemaConcettuale](../Data%20Warehouse/img/02_DSTBD_ProgettazioneMadeInItaly.png)

## Progettazione Logica

**Esportazioni**(*IdTempo, IdAzienda, IdStato, IdVino, IdConfezione*, TotLitri, TotPrezzo, DimOrdine)

**Tempo** (*IdTempo*, Mese, 2M, 3M, 4M, 6M, Anno)

**Azienda** (*IdAzienda*, Provincia, Regione, AreaGeografica)

**Destinazione** (*IdStato*, Stato, Continente)

**TipoVino** (*IdVino*, DOC, DOP, DOCG)

**Confezione** (*IdConfezione*, TipoConfezione)

"La dimensione DimOrdine ha pochi valori distinti ==> usiamo la tecnica di push down"

1. Considerando solo i vini che hanno la certificazione “DOC” esportati in Asia, selezionare il prezzo medio al litro per ogni anno, la percentuale di litri esportati nell’anno rispetto al totale di tutti gli anni, e il totale cumulativo annuale di litri esportati. Si efettui l’analisi separatamente per ogni tipo di confezionamento.

In [ ]:
SELECT C.TipoConfezione, T.Anno,
    SUM(E.TotPrezzo) / SUM(E.TotLitri) AS PrezzoMedioAlLitro,
    100*SUM(TotLitri) / SUM(SUM(TotLitri)) OVER(
        PARTITION BY C.TipoConfezione
    ) AS PercLitriEsportati,
    SUM(SUM(TotLitri)) OVER(
        PARTITION BY C.TipoConfezione
        ORDER BY T.Anno
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS TotCumulativoAnnuale
FROM Esportazioni E
JOIN Destinazione D ON D.IdStato = E.IdStato
JOIN TipoVino V ON V.IdVino = E.IdVino
JOIN Tempo T ON T.IdTempo = E.IdTempo
JOIN Confezione C ON C.IdConfezione = E.IdConfezione
WHERE V.DOC = 1 AND D.Continente = 'Asia'
GROUP BY C.TipoConfezione, T.Anno;

2. Per ogni regione, selezionare il prezzo medio al litro, il numero medio di litri di vino esportati per provincia, e la percentuale di litri di vino esportati da ogni regione rispetto al totale dell’area geografica di appartenenza (Nord, Centro, Sud). Si efettui l’analisi separatamente per ogni anno.

In [ ]:
SELECT T.Anno, A.Regione, A.AreaGeografica 
    SUM(TotPrezzo) / SUM(TotLitri) AS PrezzoMedioAlLitro,
    SUM(TotLitri) / COUNT (DISTINCT A.Provincia) AS AvgLitriMediPerProvincia,
    100*SUM(TotLitri) / SUM(SUM(TotLitri)) OVER(
        PARTITION BY A.AreaGeografica, T.Anno
    ) AS PercSuAreaGeografica
FROM Esportazioni E 
JOIN Azienda A ON A.IdAzienda = E.IdAzienda
JOIN Tempo T ON T.IdTempo = E.IdTempo
GROUP BY T.Anno, A.Regione, A.AreaGeografica; 

3. Assegnare un rank ad ogni stato estero per quantità totale di vino importata. Efettuare l’analisi separatamente per ogni tipo di vino. Si considerino solo gli ordini di grandi dimensioni.

In [ ]:
SELECT V.TipoVino, D.Stato, SUM(E.TotLitri) AS TotLitriImportati,
     RANK() OVER(
        PARTITION BY V.TipoVino
        ORDER BY SUM(E.TotLitri) DESC
     ) AS PosRank
FROM Esportazioni E 
JOIN TipoVino V ON V.IdVino = E.IdVino
JOIN Destinazione D ON D.IdStato = E.IdStato
WHERE E.DimOrdine = 'Grande'
GROUP BY V.TipoVino, D.Stato;